# 04 — Modeling and Evaluation

## Objective

Predict next-day streamflow using three beginner-readable approaches:

1. **Persistence baseline:** tomorrow's flow is approximately today's flow.
2. **Linear Regression:** a transparent linear benchmark.
3. **Random Forest Regressor:** a nonlinear model that can represent interactions.

The first 80% of dates form the training set and the final 20% form the test set. The time series is never shuffled.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from train_models import FEATURES, TARGET, train_models

features = pd.read_csv(
    ROOT / "data" / "processed" / "hydroclimate_features.csv",
    parse_dates=["date"],
).sort_values("date").reset_index(drop=True)
features.shape

(349, 15)

## 1. Chronological 80/20 split

Random splitting would let the model learn from later seasons and then evaluate on earlier dates. The chronological split better represents predicting an unseen future period.

In [2]:
split_index = int(len(features) * 0.80)
train = features.iloc[:split_index]
test = features.iloc[split_index:]

split_summary = pd.DataFrame({
    "set": ["Train", "Test"],
    "rows": [len(train), len(test)],
    "start": [train.date.min().date(), test.date.min().date()],
    "end": [train.date.max().date(), test.date.max().date()],
})
split_summary

,set,rows,start,end
0,Train,279,2020-01-08,2020-10-21
1,Test,70,2020-10-22,2020-12-30


## 2. Leakage check

- The target is created by shifting streamflow one day backward into the current row.
- Lagged flow features are shifted before rolling.
- All predictors are known by the end of the forecast date.
- Models are fit only on training rows.
- No random shuffle or future target value is used as a feature.

Today’s streamflow is included because it is also the information used by the persistence baseline, making the comparison fair.

In [3]:
assert train.date.max() < test.date.min()
assert TARGET not in FEATURES
assert features[FEATURES + [TARGET]].isna().sum().sum() == 0
print("Leakage checks passed.")

Leakage checks passed.


## 3. Train and evaluate

MAE is the average absolute error, RMSE penalizes large misses more strongly, and R² measures variance explained relative to predicting the test mean. With one year of data and only 70 test days, these metrics are illustrative rather than definitive.

In [4]:
model_metrics, predictions, importance = train_models(features, persist=True)

comparison = pd.DataFrame([
    {"Model": "Persistence Baseline", **model_metrics["persistence_baseline"]},
    {"Model": "Linear Regression", **model_metrics["linear_regression"]},
    {"Model": "Random Forest", **model_metrics["random_forest"]},
]).set_index("Model")
comparison.round(3)

               model    MAE    RMSE    R2
Persistence Baseline 53.153 109.597 0.426
   Linear Regression 60.863  83.974 0.663
       Random Forest 51.564  92.586 0.590


,MAE,RMSE,R2
Model,,,
Persistence Baseline,53.153,109.597,0.426
Linear Regression,60.863,83.974,0.663
Random Forest,51.564,92.586,0.590


## 4. Actual versus predicted

The plot shows performance only on the held-out final 20% of dates. Large peak-flow misses are particularly visible because RMSE is sensitive to them.

In [5]:
ax = predictions.set_index("date")[[
    TARGET, "linear_prediction", "random_forest_prediction"
]].rename(columns={
    TARGET: "Actual", "linear_prediction": "Linear Regression",
    "random_forest_prediction": "Random Forest"
}).plot(figsize=(12, 5))
ax.set(title="Held-Out Next-Day Streamflow: Actual vs Predicted", ylabel="Streamflow (m³/s)")
plt.tight_layout()
plt.show()

C:\Users\adith\AppData\Local\Temp\ipykernel_45240\3813418827.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Random Forest feature importance

Importance measures predictive usefulness within this fitted model; it does not show causation. Correlated lagged and rolling features can divide importance among themselves.

In [6]:
top_importance = importance.head(10).sort_values("importance")
ax = top_importance.plot.barh(x="feature", y="importance", legend=False, figsize=(9, 5), color="#2a9d8f")
ax.set(title="Random Forest Feature Importance", xlabel="Importance", ylabel="")
plt.tight_layout()
plt.show()
importance.head(10)

C:\Users\adith\AppData\Local\Temp\ipykernel_45240\2626326160.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,importance
0,streamflow_cms,0.850078
2,precipitation_mm,0.055588
12,day_of_year,0.016447
6,temp_3day_mean,0.013100
1,mean_temp_c,0.012037
8,streamflow_lag1,0.011321
3,precip_lag1,0.009273
10,streamflow_7day_mean,0.007951
9,streamflow_lag3,0.006971
4,precip_3day,0.005812


## Interpretation and limitations

Linear Regression achieves the strongest held-out RMSE and R², while Random Forest has the lowest MAE. The persistence baseline remains competitive, confirming strong day-to-day river persistence. Today’s flow is the dominant Random Forest feature, with precipitation and seasonal variables adding information.

Important limitations are the single-year sample, a test period concentrated in late autumn and winter, rare extreme flows, nearby-airport climate measurements that may not represent the full watershed, missing snowpack/soil-moisture/regulation variables, and interpolation performed for retrospective analysis. This is a portfolio study, not an operational flood forecasting system.